# Image Overlay Walkthough

### Use this code if you want to produce a GIF of the colorful FRET/CFP video and the PIV analysis

If you have read the previous image analysis code or the directional data code most of this is repeated information.


First, we declare the **libraries**.

In [ ]:
import os
import csv
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
from skimage.io import imread
from skimage.util import img_as_float
from skimage.filters import gaussian, threshold_otsu
from openpiv import pyprocess, validation, filters

Next, we identify our **paths** to our images for both the **RFP** and **ratio** data.

In [ ]:
rfp_path_pattern = "/Volumes/qiongy-data/Users/Lucy/Data/04292026FreshEnergyInterphaseCycling/Pos14/img_*_7-RFP-T_000.tif"
fret_path_pattern = "/Volumes/qiongy-data/Users/Maggie/OLD 04-29-26 PIV results/ImageJdata/POS14ratiodata/color/IMD_2.1_00*.tif"

#can add [:-###] if you want to remove x number of frames from the back end
rfp_paths = sorted(glob(rfp_path_pattern))
fret_paths = sorted(glob(fret_path_pattern))

We double check that everything lines up.

We also intitalize some of our variables and run a print check.

In [ ]:
num_frames = min(len(rfp_paths), len(fret_paths))
if num_frames < 2:
    raise ValueError(f"Insufficient matching frames found! RFP: {len(rfp_paths)}, FRET: {len(fret_paths)}")

rfp_paths = rfp_paths[:num_frames]
fret_paths = fret_paths[:num_frames]
total_pairs = num_frames - 1

print(f"Synchronized pipelines: Processing {total_pairs} paired frames ({num_frames} total frames).")

Next, we delcare our time step, window size and PIV vector scale.

In [ ]:
dt = 3.0  # 3 minutes between frames

#size of the window we are looking at 
winsize = 32 
searchsize = 32  
overlap = 8 

#hard setting the velocity scale, mostly for asetetics 
vmin, vmax = 0.0, 5.0  # Vector velocity scale

Identifying the size of our image from the FRET/CFP data.

In [ ]:
first_fret_raw = imread(fret_paths[0])
fret_height, fret_width = first_fret_raw.shape[:2]

Create some temp holding spaces and create our **bins**.

In [ ]:
#this is for efficentcy, I suck at spelling holy 
temp_dir = "combined_temp_frames"
os.makedirs(temp_dir, exist_ok=True)

#declaring our lists where the data will be stored
time_list = []
avg_speed_list = []
max_speed_list = []
global_fret_activity = []
frame_files = []

kymo_x_wave_T = np.zeros((total_pairs, fret_width))  
kymo_y_wave_T = np.zeros((total_pairs, fret_height)) 

We also declare some of the presentation stuff before this loop.

In [ ]:
#presentation stuff
fig, ax = plt.subplots(figsize=(8, 8))
#this us for the arrows, see other code for reference
dummy_Q = ax.quiver([0], [0], [0], [0], [0], cmap="plasma", scale=115, clim=(vmin, vmax))
cb = fig.colorbar(dummy_Q, ax=ax, orientation='horizontal', pad=0.08)
cb.set_label('Mechanical Velocity Magnitude')

We start the **loop**:

In [ ]:
for i in range(total_pairs):
    if (i + 1) % 10 == 0 or i == 0 or i == total_pairs - 1:
        print(f"Processing paired frame {i + 1} of {total_pairs}...")

The rest is pretty standard again. 

We append the time.

Initalize our current and next frame.

And convert the color to B&W for processing 

In [ ]:
    current_time = i * dt
    time_list.append(current_time)

    curr_rfp = imread(rfp_paths[i])
    next_rfp = imread(rfp_paths[i + 1])
    
    raw_fret_curr = imread(fret_paths[i])
    raw_fret_next = imread(fret_paths[i + 1])

    #converting color to  B&W
    if raw_fret_curr.ndim == 3:
        fret_curr = img_as_float(raw_fret_curr[:, :, 0])
        fret_next = img_as_float(raw_fret_next[:, :, 0])
    else:
        fret_curr = img_as_float(raw_fret_curr)
        fret_next = img_as_float(raw_fret_next)

    fret_curr = np.clip(np.nan_to_num(fret_curr), 0.0, 2.0)
    fret_next = np.clip(np.nan_to_num(fret_next), 0.0, 2.0)

We then analyze the **absolute change** for the **FRET** data. 

In [ ]:
    #fret data 
    fret_diff = np.abs(fret_next - fret_curr)
    global_fret_activity.append(np.mean(fret_diff))
    kymo_x_wave_T[i, :] = np.median(fret_diff, axis=0)
    kymo_y_wave_T[i, :] = np.median(fret_diff, axis=1)

Next, we look at the **RFP** (mechanical) channel. 

We blur based on the **guassian** and then create our mask from the mechanical channel since that is where the **PIV** analysis is being done. 

In [ ]:
#RFP data 
    rfp_curr_float = img_as_float(curr_rfp)
    rfp_next_float = img_as_float(next_rfp)

    rfp_curr_blur = gaussian(rfp_curr_float, sigma=1.0)
    rfp_next_blur = gaussian(rfp_next_float, sigma=1.0)

    otsu_curr = threshold_otsu(rfp_curr_blur)
    otsu_next = threshold_otsu(rfp_next_blur)

    rfp_curr_prep = (np.where(rfp_curr_blur > otsu_curr, rfp_curr_float, 0.0) * 255).astype(np.int16)
    rfp_next_prep = (np.where(rfp_next_blur > otsu_next, rfp_next_float, 0.0) * 255).astype(np.int16)

    mask_layer = (rfp_curr_blur <= otsu_curr)

Declaring our **changes** and **catching errors**. 

In [ ]:
     u, v, sig2noise = pyprocess.extended_search_area_piv(
        rfp_curr_prep,
        rfp_next_prep,
        window_size=winsize,
        overlap=overlap,
        dt=dt,
        search_area_size=searchsize,
        sig2noise_method='peak2peak',
    )

    x, y = pyprocess.get_coordinates(image_size=rfp_curr_prep.shape, search_area_size=searchsize, overlap=overlap)
    flags = validation.sig2noise_val(u, v, sig2noise, threshold=1.0015)
    u, v = filters.replace_outliers(u, v, flags, method='localmean', max_iter=10, kernel_size=2)

    u, v = u.astype(float), v.astype(float)

Create our **mask**.

In [ ]:
grid_mask = mask_layer[y.astype(int), x.astype(int)]
    
    GIGAMASK = grid_mask | np.isnan(u) | np.isnan(v)
    masked_u = np.ma.masked_array(u, mask=GIGAMASK)
    masked_v = np.ma.masked_array(v, mask=GIGAMASK)

Run the necessary **calculations**.

In [ ]:
    magnitude = np.ma.sqrt(masked_u**2 + masked_v**2)

    avg_speed_list.append(np.nanmean(magnitude))
    max_speed_list.append(np.nanmax(magnitude))

Housekeeping!

In [ ]:
ax.clear()
    ax.imshow(raw_fret_curr)
    Q = ax.quiver(
        x, y, 
        masked_u, -masked_v,
        magnitude,
        cmap="plasma", 
        scale=115, 
        width=0.005
    )
    Q.set_clim(vmin, vmax)
    ax.set_xlim(0, fret_width)
    ax.set_ylim(fret_height, 0) 
    ax.set_title(f"PIV Vectors over Raw FRET/CFP Image | Minute: {int(current_time)}")
    frame_path = os.path.join(temp_dir, f"overlay_{i:03d}.png")
    plt.savefig(frame_path, dpi=100)
    frame_files.append(frame_path)
plt.close()

Finally, we print and save our results. 

In [ ]:
#data output stuff 
print("\nStitching synchronized overlay movie...")
images = [iio.imread(f) for f in frame_files]
gif_output_path = "/Users/maggie/Desktop/mechanical_biochemical_overlayRAWMAPPOS14.gif"
iio.imwrite(gif_output_path, images, plugin="pillow", duration=250, loop=0)

#csv output, can always add more data to this in the future depending on what we want to see 
csv_filename = "/Users/maggie/Desktop/synchronized_wave_analyticsPOS14.csv"
print(f"Saving compiled spatial metrics to {csv_filename}...")
with open(csv_filename, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["Time(min)", "AvgMechanicalSpeed", "MaxMechanicalSpeed", "GlobalBiochemicalFRETDelta"])
    writer.writerows(zip(time_list, avg_speed_list, max_speed_list, global_fret_activity))

#more cleanin 
print("Cleaning up temp files...")
for f in frame_files:
    os.remove(f)
os.rmdir(temp_dir)

#indcated end of code
#another print check 
print("\nPipeline execution complete! Overlay saved as 'mechanical_biochemical_overlay.gif'")

# End of Code!

### Some Notes: 

While you can use this to generate some data, I would suggest using the indivialized PIV and Kymograph codes for image analysis. This code was mainly made to create a pretty visual to see how the PIV activity might be relating to Biohcemical activity. 

But, you can add to this and make it more of an indepth analysis if you would like!